# Grid processing: geometry, mask, query, regridding

This notebook shows how to use some extra features for your gridmarthe treatments


In [ ]:
# import modules
import numpy as np
import gridmarthe as gm
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [ ]:
# load data for example
grid = gm.load_marthe_grid('./data/craie_npc.permh', drop_nan=True)  # NaN needs to be drop or set (change 0 to nan for permh for example)
grid

## Query grid

`gridmarthe` load data as a spatially reduced grid (spatial dimension is reduced to 1D).
Hence, `x` and `y` coordinates are stored as variables, attached to grid indices, and not
as dimensions. It is then not possible to query the grid with `x` and `y` coordinates
and `xarray.Dataset.sel()` method.

To query the grid with `x` and `y` coordinates, one can use the :py:func:`gridmarthe.assign_coords`
function to set coordinates as dimension, or use the custom function :py:func:`gridmarthe.sel_by_coords`
to query the grid with `x` and `y` coordinates directly on 1D spatial array.

In [ ]:
subset = gm.sel_by_coords(grid, x=(586000, 646000), y=(2593100, 2633600))
subset

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10, 3.5))
gm.plot_nested_grid(gm.assign_coords(grid).sel(z=6).squeeze('time'), ax=ax[0], norm=LogNorm())
gm.plot_nested_grid(gm.assign_coords(subset).sel(z=6).squeeze('time'), ax=ax[1], norm=LogNorm())
ax[0].set_title('Full data set')
ax[1].set_title('Subset by coords')
fig.tight_layout()

For convenience and interoperability with other tools, user can also search
for the `zone` index of a particular cell point, using either x, y coordinates
or the `i` (column) and `j` (row) indices (:py:func:`gridmarthe.search_zone`).

In [ ]:
ds = gm.load_marthe_grid("data/chasim_hallue.out", add_col_row=True).isel(time=0)
# search with column/row indices:
idx = gm.search_zone(ds, i=23, j=32)
print(idx)

## Get surface mask

See :py:func:`gridmarthe.get_surface_layer` for more details.

In [ ]:
surf = gm.get_surface_layer(grid)
surf

In [ ]:
# xarray version
toto = gm.assign_coords(surf, add_lay=False) # add lay false because it needs to be a variable to plot, not a dimension
gm.plot_outcrop(toto)
plt.show()

This function also allow user to get the value of a variable in the surface layer,
or a subset of aquifer layers.

For example, to get the value of your variable (groundwater head for example) in
the first layer encountered between 3 layers:

In [ ]:
head_surf = gm.get_surface_layer(grid, aquif_layers=[6,8,9])  # get the values in the first layer between layers 6, 8 and 9

## Get mask of active domain

See :py:func:`gridmarthe.get_active_mask`

In [ ]:
mask = gm.get_active_mask(grid) # return a geopandas geodataframe

In [ ]:
mask.boundary.plot(color='k')
plt.show()

## Compute depths and thickness

Users can get geometry attributes (depth, thickness, upper/lower altitudes) in a dataset:

See [compute_geometry()](:py:func:`gridmarthe.compute_geometry`) for more details.

In [ ]:
topo = gm.load_marthe_grid('./data/example.topog')   # be careful with nan here, topography can be set outside of active area
hsubs = gm.load_marthe_grid('./data/example.hsubs')  # same here, hsubs can be set outside of active area
geom = gm.compute_geometry(topo, hsubs)
# print(geom.where(~geom['depth'].isnull(), drop=True))
geom


## Interpolation, regridding, transformations

### Regrid (coarse/refine resolution)

Utils function to coarse/refine grid resolution are provided, wrapping
`xarray.DataArray.interp` method.
By default, a linear interpolation is used.

See [docs.xarray.dev/interpolation](https://docs.xarray.dev/en/stable/user-guide/interpolation.html)
for more details

In [ ]:
ds_coarse = gm.rescale_grid(grid, res=1e3)  # kwargs can be passed to interp method
ds_8km = gm.rescale_grid(grid, res=8e3)
ds_100m = gm.rescale_grid(grid, res=1e2)

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(10,8))
gm.plot_nested_grid(grid, ax=ax[0,0], norm=LogNorm(), itime=0, layer=6)
gm.plot_nested_grid(ds_coarse, ax=ax[0,1], norm=LogNorm(), itime=0, layer=6)
gm.plot_nested_grid(ds_8km, ax=ax[1,0], norm=LogNorm(), itime=0, layer=6)
gm.plot_nested_grid(ds_100m, ax=ax[1,1], norm=LogNorm(), itime=0, layer=6)
ax[0,0].set_title('Original grid, 500m')
ax[0,1].set_title('Regrid to 1km')
ax[1,0].set_title('Regrid to 8km')
ax[1,1].set_title('Regrid to 100m')
fig.tight_layout()

### Projection transformation

**Warning**: this function is still experimental and only works for regular grids,
read without dropping NaNs and invalid data (a full x, y, dx, dy grid is required).


In [ ]:
ds_l2e = gm.load_marthe_grid('data/chasim_hallue.out', xyfactor=1e3).isel(time=0)
ds_l93 = gm.reproj_grid(ds_l2e, from_epsg='EPSG:27572', to_epsg='EPSG:2154', decimals=0)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(7, 3))
gm.assign_coords(ds_l2e)['charge'].where(lambda x: x < 9999.).plot.pcolormesh(ax=ax[0])
gm.assign_coords(ds_l93)['charge'].where(lambda x: x < 9999.).plot.pcolormesh(ax=ax[1])
fig.tight_layout()
ax[0].set_title('Lambert II étendu')
ax[1].set_title('Lambert 93')
plt.show()